# Projection baselines

**Question:** How well can transparent historical rules predict a future player-season?  
**Data:** Canonical regular-season `player_week_stats`, aggregated into `player_season_features`; the worked example below is labeled synthetic so the notebook runs before the Phase 3 build.  
**Unit of observation:** One player and one prediction season.  
**Target:** Next-season fantasy points per game for this demonstration; positive-snap `games_active` and total points are separate production targets.  
**Feature cutoff:** A row with feature season `t` may use only seasons `t` and earlier to predict `t + 1`.  
**Validation:** Expanding, chronological prediction-season folds; never a random row split.  
**Interpretation caveat:** These baselines are reference forecasts, not final rankings, and an observed weekly row is not automatically proof of a game played. Phase 3 has not begun machine-learning training.

## Prerequisites and local warehouse

Run the following from the repository root when the Phase 3 feature builder is available:

```powershell
fantasy-draft data load-nflverse
fantasy-draft data load-nflverse-participation
fantasy-draft features build-player-seasons --prediction-season 2026
fantasy-draft data audit
```

The next cell opens DuckDB read-only. It previews generated rows when they exist and otherwise explains the prerequisite without failing.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


project_root = find_project_root()
warehouse_path = project_root / "data" / "warehouse" / "fantasy_football.duckdb"
print(f"Project: {project_root}")
print(f"Warehouse: {warehouse_path}")

In [ ]:
feature_preview = pd.DataFrame()
if not warehouse_path.is_file():
    print("Warehouse not found. Run: fantasy-draft data init-warehouse")
else:
    with duckdb.connect(str(warehouse_path), read_only=True) as connection:
        table_names = {
            row[0]
            for row in connection.execute(
                "SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'"
            ).fetchall()
        }
        if "player_season_features" not in table_names:
            print("Feature table not initialized. Run the Phase 3 prerequisite commands above.")
        else:
            feature_count = connection.execute(
                "SELECT count(*) FROM player_season_features"
            ).fetchone()[0]
            print(f"player_season_features rows: {feature_count:,}")
            if feature_count:
                feature_preview = connection.execute(
                    "SELECT * FROM player_season_features "
                    "ORDER BY prediction_season, position, player_id LIMIT 10"
                ).fetchdf()
            else:
                print("The table is empty. Build and validate features before training baselines.")
feature_preview

## Time semantics and participation

A source manifest's acquisition time proves when the local file was captured. It is not the feature cutoff. A historical feature row must instead enforce that every contributing football season is earlier than its prediction season, while preserving manifest provenance for reproducibility.

Likewise, `observed_weeks` is not automatically `games_active`. Production points per game uses mapped positive-snap participation. If a nonzero-stat game lacks that evidence, the player-season denominator and points per game remain unavailable instead of silently inventing precision.

Position identity follows the same cutoff discipline. A later career position label cannot be copied backward into a historical rookie or other entry cohort. When a historical candidate has no cutoff-safe, time-versioned position evidence, the production builder excludes and reports that candidate. The player-identity snapshot acquired in August 2026 predates the September 1, 2026 live cutoff, so its static position is a safe fallback for live 2026 candidates only; it is too late for historical folds.

As a result, historical rookie-baseline performance cannot be measured honestly and comprehensively until the project adds a historical preseason-position archive. The quality report preserves the excluded-entry-cohort count instead of hiding this limitation.

## A deterministic worked example

This small fixture is synthetic and is never production training data. Each 2024 prediction uses only player history through 2023. It demonstrates four of the five production baselines: previous-season, weighted-history, age-adjusted, and position-shrinkage. The production-only weighted-components baseline needs passing/rushing/receiving inputs that this compact points-per-game fixture intentionally omits.

In [ ]:
history = pd.DataFrame(
    [
        ("WR_A", "WR", 2021, 23, 11.0, 16),
        ("WR_A", "WR", 2022, 24, 13.0, 17),
        ("WR_A", "WR", 2023, 25, 15.0, 17),
        ("WR_A", "WR", 2024, 26, 14.0, 16),
        ("WR_B", "WR", 2021, 28, 17.0, 17),
        ("WR_B", "WR", 2022, 29, 16.0, 16),
        ("WR_B", "WR", 2023, 30, 13.0, 12),
        ("WR_B", "WR", 2024, 31, 11.0, 13),
        ("RB_A", "RB", 2021, 22, 10.0, 14),
        ("RB_A", "RB", 2022, 23, 12.0, 16),
        ("RB_A", "RB", 2023, 24, 14.0, 17),
        ("RB_A", "RB", 2024, 25, 15.0, 17),
        ("RB_B", "RB", 2021, 27, 15.0, 15),
        ("RB_B", "RB", 2022, 28, 13.0, 13),
        ("RB_B", "RB", 2023, 29, 9.0, 6),
        ("RB_B", "RB", 2024, 30, 8.0, 8),
    ],
    columns=["player_id", "position", "season", "age", "points_per_game", "games"],
).sort_values(["player_id", "season"], ignore_index=True)
history

In [ ]:
feature_season = 2023
prediction_season = feature_season + 1
weights = {0: 0.60, 1: 0.30, 2: 0.10}
shrinkage_strength = 8.0
age_adjustments = {
    ("RB", "under_27"): 0.4,
    ("RB", "27_plus"): -1.2,
    ("WR", "under_30"): 0.2,
    ("WR", "30_plus"): -0.8,
}


def age_bucket(position: str, age: int) -> str:
    threshold = 27 if position == "RB" else 30
    return f"under_{threshold}" if age < threshold else f"{threshold}_plus"


rows = []
for player_id, player_history in history.groupby("player_id", sort=True):
    allowable = player_history[player_history["season"] <= feature_season].copy()
    target = player_history.loc[
        player_history["season"] == prediction_season, "points_per_game"
    ]
    if allowable.empty or target.empty:
        continue
    latest = allowable.iloc[-1]
    weighted_values = []
    for lag, weight in weights.items():
        value = allowable.loc[
            allowable["season"] == feature_season - lag, "points_per_game"
        ]
        if not value.empty:
            weighted_values.append((float(value.iloc[0]), weight))
    weighted_prediction = sum(value * weight for value, weight in weighted_values) / sum(
        weight for _, weight in weighted_values
    )
    position_mean = history.loc[
        (history["position"] == latest["position"])
        & (history["season"] <= feature_season),
        "points_per_game",
    ].mean()
    shrunk = (
        latest["games"] * latest["points_per_game"]
        + shrinkage_strength * position_mean
    ) / (latest["games"] + shrinkage_strength)
    bucket = age_bucket(str(latest["position"]), int(latest["age"]))
    rows.append(
        {
            "player_id": player_id,
            "position": latest["position"],
            "feature_season": feature_season,
            "prediction_season": prediction_season,
            "actual": float(target.iloc[0]),
            "previous": float(latest["points_per_game"]),
            "weighted": weighted_prediction,
            "age_adjusted": weighted_prediction
            + age_adjustments[(str(latest["position"]), bucket)],
            "shrinkage": float(shrunk),
        }
    )

predictions = pd.DataFrame(rows)
assert (predictions["feature_season"] < predictions["prediction_season"]).all()
predictions

The age adjustments above are fixed teaching values, not estimates for production use. In a real fold, derive each adjustment only from that fold's earlier training seasons. The shrinkage volume is labeled `games` because the synthetic fixture defines it that way; production code must use verified positive-snap participation. The fifth production baseline independently weights per-active-game stat components and sends the reconstructed line through the league scoring engine. Weekly threshold-bonus points are averaged separately, so a bonus is never applied to an averaged yardage line.

In [ ]:
def regression_metrics(frame: pd.DataFrame, prediction_column: str) -> dict[str, float]:
    errors = frame[prediction_column] - frame["actual"]
    return {
        "mae": float(errors.abs().mean()),
        "rmse": float((errors.pow(2).mean()) ** 0.5),
        "median_absolute_error": float(errors.abs().median()),
        "spearman": float(
            frame[prediction_column].rank(method="average").corr(
                frame["actual"].rank(method="average")
            )
        ),
    }


metric_rows = []
for baseline in ["previous", "weighted", "age_adjusted", "shrinkage"]:
    metric_rows.append({"baseline": baseline, **regression_metrics(predictions, baseline)})
metrics = pd.DataFrame(metric_rows).sort_values(["mae", "baseline"], ignore_index=True)
metrics

In [ ]:
def top_n_overlap_by_position(
    frame: pd.DataFrame, prediction_column: str, top_n: int = 1
) -> pd.DataFrame:
    rows = []
    for position, group in frame.groupby("position", sort=True):
        size = min(top_n, len(group))
        projected = set(group.nlargest(size, prediction_column)["player_id"])
        actual = set(group.nlargest(size, "actual")["player_id"])
        rows.append(
            {
                "position": position,
                "baseline": prediction_column,
                "top_n": size,
                "overlap": len(projected & actual) / size,
            }
        )
    return pd.DataFrame(rows)


pd.concat(
    [top_n_overlap_by_position(predictions, baseline) for baseline in metrics["baseline"]],
    ignore_index=True,
)

## Expanding folds

The fold boundaries below are expressed as prediction seasons. For example, validating prediction season 2020 means its feature row stops at 2019. A production evaluator must also fit age adjustments, shrinkage parameters, tier boundaries, and any other learned choices inside each training fold.

In [ ]:
def expanding_folds(
    first_training_season: int, validation_seasons: list[int], test_season: int
) -> pd.DataFrame:
    rows = []
    for evaluation_season in [*validation_seasons, test_season]:
        rows.append(
            {
                "train_prediction_seasons": f"{first_training_season}-{evaluation_season - 1}",
                "evaluation_prediction_season": evaluation_season,
                "role": "test" if evaluation_season == test_season else "validation",
                "latest_allowed_feature_season": evaluation_season - 1,
            }
        )
    folds = pd.DataFrame(rows)
    assert (
        folds["latest_allowed_feature_season"]
        < folds["evaluation_prediction_season"]
    ).all()
    return folds


folds = expanding_folds(2016, [2020, 2021, 2022, 2023, 2024], 2025)
folds

## Leakage checklist and exercise

Before trusting a report, verify that target-season statistics, post-draft ADP, final injuries, final rosters, target-derived tiers, and identity snapshots acquired after a row's cutoff cannot enter features. Confirm regular-season filtering, target exclusion, manifest provenance, deterministic rebuilds, missing-target counts, reported entry-cohort position exclusions, and chronological fold order.

**Exercise:** Change the history weights to 50%/30%/20% and increase `shrinkage_strength`. Predict which low-volume player will move most. Then compare MAE, RMSE, median absolute error, Spearman correlation, and top-N overlap. Explain why the lowest-RMSE setting need not produce the best positional ranking.